# 03 · Endole sample (aggregator)

Research log: [Step 3](../docs/research_log.md#step-3--primul-caz-real-și-adresele-comune), [Step 4](../docs/research_log.md#step-4--endole-pe-30-de-firme) (in Romanian). Data and checks as of October 2026.

**Question.** For how many large (group-4) companies does the Endole aggregator show a VAT number, and is it correct?

**Method.** Endole links for the first 12-company group-4 set, then a random sample of 30 group-4 companies, checked by hand on Endole and on the HMRC web checker. Also: SIC goods / import-export filter and the VAT mod-97 checksum function.

**Result (from the research log).**
- Endole shows a VAT number for 13/30 (43%, 95% Wilson CI 27–61%); 13/13 are valid at HMRC on the exact company name; 9/13 also match the postcode.
- Endole Terms (2.5) forbid automation, so it is not a usable source for a product.
- Checksum accepts 13/13 confirmed numbers and rejects `123456789`; ~2% of random 9-digit numbers pass. SIC goods proxy: 19.0% of all companies (upper bound).

**Notes for re-running.**
- The cell that builds the Endole links writes an **empty template** to `../data/urma_endole_30.csv` (gitignored). The completed trail is `../data/hmrc_checks/urma_endole_30.csv`.
- The SIC-filter cell reads the whole CH file and writes `../data/firme_probabil_import_export.csv` (147 MB, not in git).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [ ]:
# Setup added for this notebook: copied verbatim from original cell(s) 14
import duckdb


In [13]:
import pandas as pd

setul = pd.read_csv('../data/samples/set_explorare_grup4.csv', dtype=str)
for _, r in setul.iterrows():
    print(r['companynumber'], r['companyname'],
          f"https://open.endole.co.uk/search/?q={r['companynumber']}")

08945805 CAYMANBRACK CAPITAL GROUP PLC https://open.endole.co.uk/search/?q=08945805
05522412 CREATIVE CARE (EAST MIDLANDS) LIMITED https://open.endole.co.uk/search/?q=05522412
07168990 GREENTECH DISTRIBUTION PLC https://open.endole.co.uk/search/?q=07168990
05667880 ARTESIAN SOLUTIONS LIMITED https://open.endole.co.uk/search/?q=05667880
03421165 DSV LEAD LOGISTICS LIMITED https://open.endole.co.uk/search/?q=03421165
07031955 FINANCIAL SERVICES GROUP LIMITED https://open.endole.co.uk/search/?q=07031955
04115071 ELLIOTT WESTLAND INSURANCE BROKERS LIMITED https://open.endole.co.uk/search/?q=04115071
SC354749 FLOW HOSPITALITY TRAINING LIMITED https://open.endole.co.uk/search/?q=SC354749
09242996 ASG PRINT HOLDINGS LIMITED https://open.endole.co.uk/search/?q=09242996
04955958 GD FIRST (UK) LIMITED https://open.endole.co.uk/search/?q=04955958
05679302 BAUFRITZ (UK) LTD. https://open.endole.co.uk/search/?q=05679302
08600289 COUGAR HOLDCO LONDON LIMITED https://open.endole.co.uk/search/?q=08600

In [14]:
import re
import pandas as pd

def link_endole(nr, nume):
    slug = re.sub(r'[^a-z0-9]+', '-', nume.lower()).strip('-')
    return f"https://open.endole.co.uk/insight/company/{nr}-{slug}"

setul = pd.read_csv('../data/samples/set_explorare_grup4.csv', dtype=str)
for _, r in setul.iterrows():
    print(link_endole(r['companynumber'], r['companyname']))

https://open.endole.co.uk/insight/company/08945805-caymanbrack-capital-group-plc
https://open.endole.co.uk/insight/company/05522412-creative-care-east-midlands-limited
https://open.endole.co.uk/insight/company/07168990-greentech-distribution-plc
https://open.endole.co.uk/insight/company/05667880-artesian-solutions-limited
https://open.endole.co.uk/insight/company/03421165-dsv-lead-logistics-limited
https://open.endole.co.uk/insight/company/07031955-financial-services-group-limited
https://open.endole.co.uk/insight/company/04115071-elliott-westland-insurance-brokers-limited
https://open.endole.co.uk/insight/company/SC354749-flow-hospitality-training-limited
https://open.endole.co.uk/insight/company/09242996-asg-print-holdings-limited
https://open.endole.co.uk/insight/company/04955958-gd-first-uk-limited
https://open.endole.co.uk/insight/company/05679302-baufritz-uk-ltd
https://open.endole.co.uk/insight/company/08600289-cougar-holdco-london-limited


In [15]:
n = 30

duckdb.sql(f"""
    WITH candidati AS (
        SELECT companyname, companynumber, accountsaccountcategory,
               siccodesictext_1, regaddresspostcode
        FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                      normalize_names = true, ignore_errors = true)
        WHERE companystatus = 'Active'
          AND companycategory = 'Private Limited Company'
          AND accountsaccountcategory IN ('FULL','GROUP','MEDIUM')
          AND LEFT(siccodesictext_1, 5) NOT IN ('99999','98000','68209','68100','64209')
          AND siccodesictext_1 <> 'None Supplied'
          AND companynumber NOT IN (
              SELECT companynumber
              FROM read_csv('../data/samples/set_explorare_grup4.csv', all_varchar = true))
    )
    SELECT * FROM candidati
    USING SAMPLE reservoir({n} ROWS) REPEATABLE (7)
""").df().to_csv('../data/samples/set_explorare_endole.csv', index=False)

In [16]:
duckdb.sql("""
    SELECT companyname, companynumber, regaddresspostcode,
           companystatus, accountsaccountcategory
    FROM read_csv('../data/BasicCompanyDataAsOneFile-2026-09-01.csv',
                  normalize_names = true, ignore_errors = true)
    WHERE companynumber = '09346879'
""").df()

,companyname,companynumber,regaddresspostcode,companystatus,accountsaccountcategory
0,ENDOLE LTD,09346879,B69 2DG,Active,TOTAL EXEMPTION FULL


In [1]:
import re
import pandas as pd

def link_endole(nr, nume):
    slug = re.sub(r'[^a-z0-9]+', '-', nume.lower()).strip('-')
    return f"https://open.endole.co.uk/insight/company/{nr}-{slug}"

setul = pd.read_csv('../data/samples/set_explorare_endole.csv', dtype=str)

urma = pd.DataFrame({
    'company_number': setul['companynumber'],
    'nume_ch':        setul['companyname'],
    'postcode_ch':    setul['regaddresspostcode'],
    'sic':            setul['siccodesictext_1'],
    'link_endole':    [link_endole(nr, nume) for nr, nume in zip(setul['companynumber'], setul['companyname'])],
})
for col in ['vat_endole', 'vrn_gasit', 'rezultat_hmrc', 'nume_hmrc', 'postcode_hmrc',
            'data_ora', 'categorie', 'timp_min', 'observatii']:
    urma[col] = ''

urma.to_csv('../data/urma_endole_30.csv', index=False)
urma[['company_number', 'nume_ch', 'link_endole']]

,company_number,nume_ch,link_endole
0,11906583,MOHARI SERVICES UK LIMITED,https://open.endole.co.uk/insight/company/1190...
1,14056015,BLUEPRINT PICTURES (STRANGERS) LTD,https://open.endole.co.uk/insight/company/1405...
2,00307876,HAULFRYN GROUP LIMITED,https://open.endole.co.uk/insight/company/0030...
3,14962760,PLEO CAPITAL UK LTD,https://open.endole.co.uk/insight/company/1496...
4,08734445,GREENSPRUCE GP LIMITED,https://open.endole.co.uk/insight/company/0873...
5,10829012,WAHED INVEST LTD,https://open.endole.co.uk/insight/company/1082...
6,06080714,WENZELS THE BAKERS LIMITED,https://open.endole.co.uk/insight/company/0608...
7,08079077,STAGE C LIMITED,https://open.endole.co.uk/insight/company/0807...
8,08961418,NEWARCH HOMES LIMITED,https://open.endole.co.uk/insight/company/0896...
9,01180133,LAINE THEATRE ARTS LIMITED,https://open.endole.co.uk/insight/company/0118...


In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

urma = pd.read_csv('../data/hmrc_checks/urma_endole_30.csv', dtype=str)
urma[['company_number', 'nume_ch', 'link_endole']]

,company_number,nume_ch,link_endole
0,11906583,MOHARI SERVICES UK LIMITED,https://open.endole.co.uk/insight/company/11906583-mohari-services-uk-limited
1,14056015,BLUEPRINT PICTURES (STRANGERS) LTD,https://open.endole.co.uk/insight/company/14056015-blueprint-pictures-strangers-ltd
2,00307876,HAULFRYN GROUP LIMITED,https://open.endole.co.uk/insight/company/00307876-haulfryn-group-limited
3,14962760,PLEO CAPITAL UK LTD,https://open.endole.co.uk/insight/company/14962760-pleo-capital-uk-ltd
4,08734445,GREENSPRUCE GP LIMITED,https://open.endole.co.uk/insight/company/08734445-greenspruce-gp-limited
5,10829012,WAHED INVEST LTD,https://open.endole.co.uk/insight/company/10829012-wahed-invest-ltd
6,06080714,WENZELS THE BAKERS LIMITED,https://open.endole.co.uk/insight/company/06080714-wenzels-the-bakers-limited
7,08079077,STAGE C LIMITED,https://open.endole.co.uk/insight/company/08079077-stage-c-limited
8,08961418,NEWARCH HOMES LIMITED,https://open.endole.co.uk/insight/company/08961418-newarch-homes-limited
9,01180133,LAINE THEATRE ARTS LIMITED,https://open.endole.co.uk/insight/company/01180133-laine-theatre-arts-limited


In [4]:
import pandas as pd

fisier = '../data/BasicCompanyDataAsOneFile-2026-09-01.csv'

# normalizeaza numele coloanelor: ' CompanyNumber' -> 'companynumber', 'SICCode.SicText_1' -> 'siccodesictext_1'
def norm(c):
    return c.strip().lower().replace('.', '').replace(' ', '')

# diviziunile SIC de "bunuri fizice"
DIVIZIUNI_BUNURI = (
    set(range(1, 4))      # agricultura, pescuit
    | set(range(5, 10))   # minerit
    | set(range(10, 34))  # productie
    | {46, 47}            # comert cu ridicata / amanuntul
    | set(range(49, 53))  # transport si logistica
)

# categoriile de conturi care apar in esantionul tau de 30
CATEGORII_ESANTION = {'FULL', 'GROUP', 'MEDIUM'}

def diviziune(sic_text):
    """Din '27900 - Manufacture of ...' scoate 27."""
    if pd.isna(sic_text) or not str(sic_text)[:2].isdigit():
        return None
    return int(str(sic_text)[:2])

def coloana_utila(c):
    n = norm(c)
    return n in ('companynumber', 'companyname', 'accountsaccountcategory') or n.startswith('siccodesictext_')

total = cu_bunuri = 0
total_filtrat = cu_bunuri_filtrat = 0
pe_diviziune = {}
rezultate = []

for bucata in pd.read_csv(fisier, dtype=str, chunksize=200_000, usecols=coloana_utila):
    bucata.columns = [norm(c) for c in bucata.columns]
    sic_cols = [c for c in bucata.columns if c.startswith('siccodesictext_')]

    div = bucata[sic_cols].apply(lambda col: col.map(diviziune))
    masca = div.isin(DIVIZIUNI_BUNURI).any(axis=1)

    # toate firmele
    total += len(bucata)
    cu_bunuri += int(masca.sum())

    # doar populatia comparabila cu esantionul
    in_populatie = bucata['accountsaccountcategory'].str.strip().str.upper().isin(CATEGORII_ESANTION)
    total_filtrat += int(in_populatie.sum())
    cu_bunuri_filtrat += int((masca & in_populatie).sum())

    for d in div['siccodesictext_1'][masca].dropna():
        pe_diviziune[int(d)] = pe_diviziune.get(int(d), 0) + 1

    rezultate.append(bucata.loc[masca, ['companynumber', 'companyname', 'accountsaccountcategory'] + sic_cols])

print('TOATE FIRMELE ACTIVE')
print(f'  Total:                  {total:,}')
print(f'  Probabil import/export: {cu_bunuri:,} ({cu_bunuri/total:.1%})\n')

print(f'DOAR CATEGORII {sorted(CATEGORII_ESANTION)} (comparabil cu esantionul)')
print(f'  Total:                  {total_filtrat:,}')
print(f'  Probabil import/export: {cu_bunuri_filtrat:,} ({cu_bunuri_filtrat/total_filtrat:.1%})\n')

print('Top 10 diviziuni (dupa primul cod SIC, toate firmele):')
for d, n in sorted(pe_diviziune.items(), key=lambda x: -x[1])[:10]:
    print(f'  {d:02d}: {n:,}')

firme = pd.concat(rezultate)
firme.to_csv('../data/firme_probabil_import_export.csv', index=False)
print(f'\nSalvat: data/firme_probabil_import_export.csv ({len(firme):,} randuri)')

TOATE FIRMELE ACTIVE
  Total:                  5,689,367
  Probabil import/export: 1,080,113 (19.0%)

DOAR CATEGORII ['FULL', 'GROUP', 'MEDIUM'] (comparabil cu esantionul)
  Total:                  121,473
  Probabil import/export: 23,337 (19.2%)

Top 10 diviziuni (dupa primul cod SIC, toate firmele):
  47: 433,740
  46: 190,179
  49: 104,903
  01: 34,492
  33: 32,969
  52: 31,075
  32: 30,128
  10: 25,438
  25: 22,305
  18: 14,083

Salvat: data/firme_probabil_import_export.csv (1,080,113 randuri)


In [5]:
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

confirmate = ['465606868','159723633','193725187','484624270','211105734','437464183',
              '324887179','230249147','143776493','811037280','208066915','315775783','264050919']
print([tva_valid_checksum(n) for n in confirmate])
print(tva_valid_checksum('123456789'))   # ar trebui False

[True, True, True, True, True, True, True, True, True, True, True, True, True]
False
